# HEP Multiagent Demo

This notebook demonstrates how to use the multi-agent framework for scientific research queries.

## Prerequisites
- Python 3.10+
- LaTeX distribution for PDF reports ([BasicTeX](https://www.tug.org/mactex/morepackages.html))
- Environment variables configured (see below)

In [1]:
# Development mode
!pip install -e ".[dev]"

# For production:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/HEP-multiagent.git


Obtaining file:///Users/celsloaner/Desktop/hep-multiagent-v2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hep-multiagent (pyproject.toml) ... done
  Created wheel for hep-multiagent: filename=hep_multiagent-0.1.0-0.editable-py3-none-any.whl size=1533 sha256=f5ecd438e8d547dcc95b64cfb6518f09ec94c6e94c56b616d47d53bba4314820
  Stored in directory: /private/var/folders/hk/qvt_rgh53jg_39prtb_3ln1m0000gp/T/pip-ephem-wheel-cache-gq76zgab/wheels/59/c5/ce/8dc962727da6a108ec3a663c3fe7ece28cb8f639498e6a786a
Successfully built hep-multiagent
  Attempting uninstall: hep-multiagent
    Found existing installation: hep-multiagent 0.1.0
    Uninstalling hep-multiagent-0.1.0:
      Successfully uninstalled hep-multiagent-0.1.0

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip insta

In [2]:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/mcp-ke.git
# mcp-ke need mcp 1.26.0 
!pip uninstall mcp_ke -y


### Set up HEP Multiagent

Set `ARGO_USER` in `.env` or pass env vars directly in server config.

In [3]:
import os
from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from hep_multiagent import Agent

load_dotenv(".env")

llm = ChatOpenAI(
    model="claudeopus46", 
    # model="claudesonnet4",
    base_url="https://apps-dev.inside.anl.gov/argoapi/v1",
    api_key=os.environ.get("ARGO_USER", "")
)

/Users/celsloaner/Desktop/hep-multiagent-v2/hep-multiagent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Initialize Agent

Create an agent with an LLM and MCP servers. Servers are auto-installed from URL at init.

In [4]:
from hep_multiagent import Agent

agent = await Agent(
    llm=llm,
    mcp_servers=[
    {
        # "url": "https://github.com/HEP-KE/mcp-ke.git@add-mcmc-paths",
        "url": "/Users/celsloaner/Desktop/mcp-ke/mcp-ke",
        "name": "mcp_ke"

        # for Nesar
        # "url": "/data/a/cpac/nramachandra/Projects/AmSC/mcp-ke"
    }
    ],
    lesson_memory=False,
)

for tool in agent.tools:
    print(f"  - {tool.name}")

  - list_agent_files
  - compute_all_models
  - compute_power_spectrum
  - compute_suppression_ratios
  - get_lcdm_params
  - get_nu_mass_params
  - get_wcdm_params
  - create_theory_k_grid
  - load_eboss_data
  - load_observational_data
  - analyze_mcmc_samples
  - compute_best_fit_power_spectrum
  - create_mcmc_corner_plot
  - create_mcmc_trace_plot
  - run_mcmc_cosmology
  - clear_session
  - compute_histogram
  - compute_percentiles
  - compute_statistics
  - delete_dataset
  - describe_dataset
  - list_datasets
  - preview_dataset
  - plot_power_spectra
  - plot_suppression_ratios
  - arxiv_agent
  - download_arxiv_paper
  - download_full_arxiv_paper
  - list_files
  - read_text_file
  - search_arxiv
  - power_spectrum_agent


In [5]:
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# OUTPUT_DIR = f"./output_hepke_arxiv_demo_{timestamp}"
OUTPUT_DIR = f"./output_hepke_demo"

result = await agent.run(
    query="""
    # Run everything via MCP tools in mcp-ke. 
    # Absolutely do not use your own code or power spectra estimations (no writing new python codes, existing tools should be used). 
    # Strictly no fake/synthetic/placeholder/realistic data..

    (1) Load observational eBOSS data.
    (2) Then compare the P(k) with wCDM, ΛCDM + Massive Neutrinos and ΛCDM (use any set of parameters you need). Plot them all.
    (3) Run a full posterior analysis using MCMC? Do this for 4 parameters (sigma8, h, Σmν or sum_nu_masses, N_species) of the ΛCDM + massive neutrinos model. 
        For the target, use the observational data from eBOSS. Go with a small run like 8 parallel chains, 100 production steps, 50 burn in steps. 
    (4) Show me the final posterior distribution plot via GetDist and the best-fit  estimates.
    """,
    output_dir=OUTPUT_DIR,
)

CancelledError: 